In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba 

# ==========================================
# 模块 A: 带对比投影头的双重注意力门 (C-STAM)
# ==========================================
class DualAttentionGate(nn.Module):
    def __init__(self, channels, embed_dim=128):
        super().__init__()
        # 1. 时间路径 (Temporal Path)
        self.temp_conv = nn.Sequential(
            nn.Conv1d(2, 1, kernel_size=7, padding=3),
            nn.Sigmoid()
        )
        self.proj_t = nn.Sequential(
            nn.Linear(1, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim)
        )
        
        # 2. 通道路径 (Channel Path)
        self.chan_mlp = nn.Sequential(
            nn.Linear(2, max(8, channels // 4)),
            nn.ReLU(inplace=True),
            nn.Linear(max(8, channels // 4), 1),
            nn.Sigmoid()
        )
        self.proj_c = nn.Sequential(
            nn.Linear(channels, embed_dim), nn.ReLU(), nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, x):
        B, C, L = x.shape
        # 时间注意力与特征提取
        t_max, _ = torch.max(x, dim=1, keepdim=True)
        t_avg = torch.mean(x, dim=1, keepdim=True)
        mt = self.temp_conv(torch.cat([t_max, t_avg], dim=1)) 
        zt = self.proj_t(mt.mean(dim=-1)) # [B, embed_dim]
        
        # 通道注意力与特征提取
        c_max, _ = torch.max(x, dim=2, keepdim=True)
        c_avg = torch.mean(x, dim=2, keepdim=True)
        mc = self.chan_mlp(torch.cat([c_max, c_avg], dim=2)).transpose(1, 2)
        zc = self.proj_c(mc.squeeze(-1)) # [B, embed_dim]

        return x * (mc * mt), zt, zc

# ==========================================
# 模块 B: 小波四向扫描桥接器 (Wavelet-Mamba Bridge)
# ==========================================
class Wavelet4WayBridge(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 定义四个方向的扫描器
        self.mamba_time_fwd = Mamba(d_model=dim, d_state=16)
        self.mamba_time_bwd = Mamba(d_model=dim, d_state=16)
        self.mamba_lead_fwd = Mamba(d_model=dim, d_state=16)
        self.mamba_lead_bwd = Mamba(d_model=dim, d_state=16)
        
        self.fusion = nn.Conv1d(dim * 4, dim, kernel_size=1)
        self.register_buffer('f_l', torch.tensor([0.707, 0.707]).view(1, 1, 2))
        self.register_buffer('f_h', torch.tensor([-0.707, 0.707]).view(1, 1, 2))

    def forward(self, x):
        B, C, L = x.shape
        # 小波分解
        cA = F.conv1d(x, self.f_l.repeat(C, 1, 1), stride=2, groups=C)
        # 1. 时间轴正反向
        x_t = cA.transpose(1, 2).contiguous()
        t_fwd = self.mamba_time_fwd(x_t).transpose(1, 2)
        t_bwd = torch.flip(self.mamba_time_bwd(torch.flip(x_t, [1])), [1]).transpose(1, 2)
        # 2. 导联轴正反向 (借用 Mamba 处理 C 维度序列)
        l_fwd = self.mamba_lead_fwd(cA)
        l_bwd = torch.flip(self.mamba_lead_bwd(torch.flip(cA, [1])), [1])
        
        refined_ca = self.fusion(torch.cat([t_fwd, t_bwd, l_fwd, l_bwd], dim=1))
        # 逆小波还原
        return F.conv_transpose1d(refined_ca, self.f_l.repeat(C, 1, 1), stride=2, groups=C) + x

# ==========================================
# 模块 C: TransSpeedFlow 精修网络 (Stage 2)
# ==========================================
class TransSpeedFlow(nn.Module):
    def __init__(self, channels=12):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(1, 128), nn.SiLU(), nn.Linear(128, channels * 2))
        self.to_q = nn.Conv1d(channels, channels, 1, groups=channels)
        self.to_k = nn.Conv1d(channels, channels, 1, groups=channels)
        self.v_gen = nn.Sequential(nn.Linear(1, 128), nn.SiLU(), nn.Linear(128, channels))
        self.final_conv = nn.Conv1d(channels, channels, 3, padding=1)

    def forward(self, t, xt, x_unet):
        B, C, L = xt.shape
        # FiLM 调制
        gamma, beta = torch.chunk(self.t_mlp(t.view(B, 1)).unsqueeze(-1), 2, dim=1)
        ft = xt * (1 + gamma) + beta
        # TimeNav Attention
        q, k = self.to_q(ft).permute(0, 2, 1), self.to_k(x_unet).permute(0, 2, 1)
        v = self.v_gen(t.view(B, 1)).unsqueeze(1).repeat(1, L, 1)
        attn = torch.softmax((q @ k.transpose(-2, -1)) * (C ** -0.5), dim=-1)
        out = (attn @ v).permute(0, 2, 1)
        return self.final_conv(out + xt)

# ==========================================
# 5. 整合主类：WaveMambaFlowNet
# ==========================================
class WaveMambaFlowNet(nn.Module):
    def __init__(self, in_ch=12):
        super().__init__()
        def block(cin, cout):
            return nn.ModuleList([
                nn.Conv1d(cin, cout, 3, padding=1),
                DualAttentionGate(cout),
                nn.Conv1d(cout, cout, 3, padding=1)
            ])

        self.enc1 = block(in_ch, 64); self.bridge1 = Wavelet4WayBridge(64)
        self.enc2 = block(64, 128);   self.bridge2 = Wavelet4WayBridge(128)
        self.bottleneck = block(128, 256)
        self.up2 = nn.ConvTranspose1d(256, 128, 2, stride=2); self.dec2 = block(256, 128)
        self.up1 = nn.ConvTranspose1d(128, 64, 2, stride=2);  self.dec1 = block(128, 64)
        self.final_u = nn.Conv1d(64, in_ch, 1)
        
        self.flow_refiner = TransSpeedFlow(channels=in_ch)

    def forward_unet(self, x):
        pairs = []
        # Encoder 1
        x1 = self.enc1[0](x)
        x1, zt1, zc1 = self.enc1[1](x1); pairs.append((zt1, zc1))
        x1 = self.enc1[2](x1)
        # Encoder 2
        x2 = self.enc2[0](F.max_pool1d(x1, 2))
        x2, zt2, zc2 = self.enc2[1](x2); pairs.append((zt2, zc2))
        x2 = self.enc2[2](x2)
        # Bottleneck
        b = self.bottleneck[0](F.max_pool1d(x2, 2))
        b, _, _ = self.bottleneck[1](b)
        b = self.bottleneck[2](b)
        # Decoder
        d2 = self.dec2[0](torch.cat([self.up2(b), self.bridge2(x2)], dim=1))
        d2, _, _ = self.dec2[1](d2)
        d2 = self.dec2[2](d2)
        d1 = self.dec1[0](torch.cat([self.up1(d2), self.bridge1(x1)], dim=1))
        d1, _, _ = self.dec1[1](d1)
        d1 = self.dec1[2](d1)
        
        return torch.tanh(self.final_u(d1)), pairs

    @torch.no_grad()
    def predict(self, x_bad, steps=30):
        self.eval()
        x_u, _ = self.forward_unet(x_bad)
        xt = x_u.clone(); dt = 1.0 / steps
        for i in range(steps):
            t = torch.full((xt.size(0),), i/steps, device=xt.device)
            xt = xt + self.flow_refiner(t, xt, x_u) * dt
        return xt